## Libraries

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl.worksheet._reader")

## Dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

file_path = '/content/1-s2.0-S0022519315005676-mmc7.xlsx'
excel_data = pd.ExcelFile(file_path)
sheet_names = excel_data.sheet_names

data_x_list, data_y_list, time_list = [], [], []
for sheet in sheet_names:
    try:
        time_value = float(sheet.rstrip('h'))
    except ValueError:
        continue
    data = pd.read_excel(file_path, sheet_name=sheet, header=None)
    x = data.iloc[2, 1:].values.astype(np.float32)
    y = data.iloc[10, 1:].values.astype(np.float32)

    data_x_list.extend(x)
    data_y_list.extend(y)
    time_list.extend([time_value] * len(x))

# Convert and normalize
data_x = torch.tensor(np.array(data_x_list) / 1e3, dtype=torch.float32).view(-1, 1).to(device)  
data_y = torch.tensor(np.array(data_y_list) * 1e6, dtype=torch.float32).view(-1, 1).to(device) 
time_stamps = torch.tensor(np.array(time_list) / 24, dtype=torch.float32).view(-1, 1).to(device) 

data_x_cpu = data_x.detach().cpu()
time_stamps_cpu = time_stamps.detach().cpu()
data_y_cpu = data_y.detach().cpu()

## Parameter Estimation

In [ ]:
class ParameterEstimator(nn.Module):
    def __init__(self, K=1.7e3):
        super().__init__()
        self.D0 = nn.Parameter(torch.tensor(np.log(1.2), dtype=torch.float32, device=device))
        self.D = nn.Parameter(torch.tensor(np.log(7.2), dtype=torch.float32, device=device))
        self.m = nn.Parameter(torch.tensor(np.log(3), dtype=torch.float32, device=device))
        self.r = nn.Parameter(torch.tensor(np.log(1.68), dtype=torch.float32, device=device))
        self.beta0 = nn.Parameter(torch.tensor(np.log(3), dtype=torch.float32, device=device))
        self.beta1 = nn.Parameter(torch.tensor(np.log(4.8), dtype=torch.float32, device=device))
        self.K = torch.tensor(K/1e4, dtype=torch.float32, device=device, requires_grad=False)

    def diffusion(self, u):
        return torch.relu(
            torch.exp(self.D0) * 0.01 +
            torch.exp(self.D) * 0.01 * torch.abs(u / torch.exp(self.K)) ** torch.exp(self.m)
        )

    def growth(self, u):
        return torch.exp(self.r) * u * (1 - u / torch.exp(self.K))

    def delay(self, t):
        return 1 / (1 + torch.exp(-(torch.exp(self.beta1) * t + torch.exp(-1 * self.beta0))))

class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, x, t):
        x = x.to(device).float()
        t = t.to(device).float()
        inp = torch.cat((x, t), dim=1)
        return self.net(inp)

def data_mse_loss(model, data_x, data_t, data_u):
    u_pred = model(data_x, data_t)
    return torch.mean((data_u / 1e6 - u_pred) ** 2)

def pde_residual_loss(model, param_estimator, x, t):
    x = x.clone().detach().requires_grad_(True)
    t = t.clone().detach().requires_grad_(True)

    u = model(x, t)
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]

    D_u = param_estimator.diffusion(u)
    D_u_u_x = D_u * u_x
    D_u_u_x_x = torch.autograd.grad(D_u_u_x, x, grad_outputs=torch.ones_like(D_u_u_x), create_graph=True, retain_graph=True)[0]

    g = param_estimator.growth(u)
    delay_t = param_estimator.delay(t)

    residual = u_t - delay_t * (D_u_u_x_x + g * u)
    return torch.mean(residual ** 2)

def compute_total_loss(model, param_estimator, data_x, data_t, data_u, x_f, t_f, lambda_pde=1.0):
    d_loss = data_mse_loss(model, data_x, data_t, data_u)
    p_loss = pde_residual_loss(model, param_estimator, x_f, t_f)
    return d_loss + lambda_pde * p_loss, d_loss, p_loss

## ARUS PINN

In [ ]:

def compute_residual_and_uncertainty(model, param_estimator, x, t):
    x = x.clone().detach().requires_grad_(True)
    t = t.clone().detach().requires_grad_(True)

    u = model(x, t)
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0]

    D_u = param_estimator.diffusion(u)
    D_u_u_x = D_u * u_x
    D_u_u_x_x = torch.autograd.grad(D_u_u_x, x, grad_outputs=torch.ones_like(D_u_u_x), create_graph=True, retain_graph=True)[0]

    g = param_estimator.growth(u)
    residual = torch.abs(u_t - D_u_u_x_x - g * u).detach().cpu().numpy().flatten()
    uncertainty = (torch.abs(u_xx) + torch.abs(u_t)).detach().cpu().numpy().flatten()
    return residual, uncertainty

def select_to_remove(x_tensor, t_tensor, R_np, U_np, eps_r, eps_u):
    mask = (R_np < eps_r) & (U_np < eps_u)
    keep_idx = np.where(~mask)[0]
    if len(keep_idx) == 0:
        return x_tensor, t_tensor
    idx_tensor = torch.tensor(keep_idx, dtype=torch.long, device=x_tensor.device)
    return x_tensor[idx_tensor], t_tensor[idx_tensor]

def select_to_add(x_test_np, t_test_np, R_np, U_np, alpha, N_add):
    eps = 1e-12
    R_max, U_max = max(R_np.max(), eps), max(U_np.max(), eps)
    score = alpha * (R_np / R_max) + (1 - alpha) * (U_np / U_max)
    idx = np.argsort(score)[-N_add:]
    return x_test_np[idx], t_test_np[idx]

def train_with_arus(pinn, param_estimator, data_x, data_t, data_u,
                    max_iters=20, epochs_per_iter=1000, N_f_initial=500,
                    N_add=400, alpha=0.5, eps_r=1e-5, eps_u=1e-5,
                    lr=1e-3, lambda_pde=1.0):

    x_min, x_max = float(data_x.min().cpu().item()), float(data_x.max().cpu().item())
    t_min, t_max = float(data_t.min().cpu().item()), float(data_t.max().cpu().item())

    x_f = (torch.rand((N_f_initial,1), device=device) * (x_max - x_min) + x_min).float()
    t_f = (torch.rand((N_f_initial,1), device=device) * (t_max - t_min) + t_min).float()

    optimizer = optim.Adam(list(pinn.parameters()) + list(param_estimator.parameters()), lr=lr)

    loss_history = []
    for iteration in range(max_iters):
        print(f"ARUS Iteration {iteration+1}/{max_iters}")
        for epoch in range(epochs_per_iter):
            optimizer.zero_grad()
            total_loss, d_loss, p_loss = compute_total_loss(pinn, param_estimator, data_x, data_t, data_u, x_f, t_f, lambda_pde)
            total_loss.backward()
            optimizer.step()
            loss_history.append(total_loss.item())
            if epoch % max(1, epochs_per_iter//5) == 0:
                print(f"Epoch {epoch}: Total={total_loss.item():.4e}, Data={d_loss.item():.4e}, PDE={p_loss.item():.4e}")

        M_test = 1000
        x_test_np = np.random.uniform(x_min, x_max, (M_test,1)).astype(np.float32)
        t_test_np = np.random.uniform(t_min, t_max, (M_test,1)).astype(np.float32)
        x_test_t = torch.tensor(x_test_np, dtype=torch.float32, device=device)
        t_test_t = torch.tensor(t_test_np, dtype=torch.float32, device=device)

        R_test, U_test = compute_residual_and_uncertainty(pinn, param_estimator, x_test_t, t_test_t)
        R_train, U_train = compute_residual_and_uncertainty(pinn, param_estimator, x_f, t_f)

        x_f, t_f = select_to_remove(x_f, t_f, R_train, U_train, eps_r, eps_u)

        x_add_np, t_add_np = select_to_add(x_test_np, t_test_np, R_test, U_test, alpha, N_add)
        x_add_t = torch.tensor(x_add_np, dtype=torch.float32, device=device)
        t_add_t = torch.tensor(t_add_np, dtype=torch.float32, device=device)
        x_f = torch.cat([x_f, x_add_t], dim=0)
        t_f = torch.cat([t_f, t_add_t], dim=0)

    return pinn, param_estimator, loss_history

pinn = PINN().to(device)
param_estimator = ParameterEstimator().to(device)

pinn, param_estimator, loss_history = train_with_arus(
    pinn, param_estimator, data_x, time_stamps, data_y,
    max_iters=30, epochs_per_iter=1000, N_f_initial=190,
    N_add=75, alpha=0.5, eps_r=0.1, eps_u=0.1, lr=1e-3, lambda_pde=1.0
)

with torch.no_grad():
    D0_val = torch.exp(param_estimator.D0).item()
    D_val = torch.exp(param_estimator.D).item()
    m_val = torch.exp(param_estimator.m).item()
    r_val = torch.exp(param_estimator.r).item()
    beta0_val = torch.exp(param_estimator.beta0).item()
    beta1_val = torch.exp(param_estimator.beta1).item()
    K_val = torch.exp(param_estimator.K).item()

    print("Final Parameter Values:")
    print(f"D0: {(1e4/24)*torch.exp(param_estimator.D0).item():.0f}")
    print(f"D: {(1e4/24)*torch.exp(param_estimator.D).item():.0f}")
    print(f"m: {torch.exp(param_estimator.m).item():.4f}")
    print(f"r: {(1/24)*torch.exp(param_estimator.r).item():.4f}")
    print(f"beta0: {torch.exp(param_estimator.beta0).item():.4f}")
    print(f"beta1: {(1/24)*torch.exp(param_estimator.beta1).item():.4f}")